# Full Pipeline: Motion Correction + Signal Extraction

Self-contained notebook that takes a raw ScanImage TIFF and produces a
`registered.zarr` containing motion-corrected movies **and** extracted
spine signals (footprints, events, denoised, F0, SNR).

Combines notebooks 01 (motion correction) and 02 (signal extraction) into
a single cell.

In [4]:
%load_ext autoreload
%autoreload 2

import logging
import os
import warnings
import time
from pathlib import Path

import numpy as np

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("pipeline")

# ============================================================
# Configuration — edit this section
# ============================================================
data_root = Path(os.environ["SPINE_TEST_DATA"])
tiff_path = data_root / "scan_00003_20240924_112852" / "scan_00003_20240924_112852.tif"
data_dir = tiff_path.parent
zarr_path = data_dir / "registered.zarr"

# Set to True to skip motion correction if registered.zarr already exists
skip_registration_if_exists = False

# Set to False to run entirely on CPU (slower but works without a GPU)
use_gpu = False

if not use_gpu:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

from spine_extraction.config import RegistrationConfig, ExtractionConfig

reg_config = RegistrationConfig(
    maxshift=50,
    clip_shift=10,
    remove_lines=4,
    ds_time=3,                  # 2^3 = 8x temporal downsample
    n_workers=1,
    init_frames=1000,
    min_cluster_size=100,
    save_full_resolution=True,  # required for NMF at full framerate
)

ext_config = ExtractionConfig(
    microscope="bergamo",
    sigma_px=1.33,
    nmf_iter=2,
    dXY=3,
    denoise_window_s=0.2,
    baseline_window_glu_s=4.0,
    tau_s=0.03,
    max_synapse_density=0.01,
    motion_thresh=2.5,
    nan_thresh=0.33,
    activity_channel=2,
)

assert tiff_path.exists(), f"TIFF not found: {tiff_path}"
t_total = time.perf_counter()

# ============================================================
# Stage 1: Motion correction
# ============================================================
from spine_extraction.io.trial_table import TrialTable, TrialEntry
from spine_extraction.io.zarr_store import ExperimentStore
from spine_extraction.registration.bergamo import register_bergamo

device = "auto" if use_gpu else "cpu"

tt = TrialTable(
    directory=str(data_dir),
    entries=[TrialEntry(filename=tiff_path.name, trial_index=1, epoch=1)],
)
store = ExperimentStore(zarr_path)

if skip_registration_if_exists and zarr_path.exists():
    try:
        store.load_alignment_data(1)
        logger.info("=== Stage 1: Motion correction SKIPPED (registered.zarr exists) ===")
    except KeyError:
        logger.info("=== Stage 1: Motion correction (zarr exists but trial 1 missing) ===")
        t0 = time.perf_counter()
        tt = register_bergamo(tt, reg_config, store, device=device)
        logger.info("Motion correction done in %.1fs", time.perf_counter() - t0)
else:
    logger.info("=== Stage 1: Motion correction ===")
    logger.info("TIFF: %s", tiff_path.name)
    logger.info("Output: %s", zarr_path)

    t0 = time.perf_counter()
    tt = register_bergamo(tt, reg_config, store, device=device)
    logger.info("Motion correction done in %.1fs", time.perf_counter() - t0)

# ============================================================
# Stage 2: Signal extraction
# ============================================================
from spine_extraction.extraction.localize import localize_sources
from spine_extraction.extraction.cross_trial_align import align_trials_cross
from spine_extraction.extraction.source_selection import select_sources
from spine_extraction.extraction.extract_trial import extract_trial
from spine_extraction.extraction.assemble import compute_denoised, censor_frames
from spine_extraction.pipeline.runner import _detect_motion_frames

logger.info("=== Stage 2: Signal extraction ===")
logger.info("Device: %s", "GPU" if use_gpu else "CPU-only")

# --- 2a. Load registered data ---
adata = store.load_alignment_data(1)
reg_ds = store.load_registered_ds(1)
n_ch = adata.num_channels
n_ds_frames = reg_ds.shape[2] // n_ch
movie_4d = reg_ds.reshape(
    reg_ds.shape[0], reg_ds.shape[1], n_ds_frames, n_ch
).transpose(0, 1, 3, 2)  # (H, W, C, T)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    mean_im = np.nanmean(movie_4d, axis=3)
h, w = movie_4d.shape[:2]

act_ch = ext_config.activity_channel - 1
movie_act = movie_4d[:, :, act_ch, :]
full_hz = 1.0 / adata.frame_time
logger.info("Image: %dx%d, %d channels, %d ds frames (%.1f Hz), full-res %.1f Hz",
            h, w, n_ch, n_ds_frames, adata.align_hz, full_hz)

# --- 2b. Source localization (on downsampled data) ---
t0 = time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    act_img, peaks = localize_sources(movie_act, ext_config, adata.align_hz)
logger.info("Localized %d sources in %.1fs", len(peaks.row), time.perf_counter() - t0)
del movie_act

# --- 2c. Cross-trial alignment (single trial = identity) ---
n_trials = 1
keep_trials = np.ones(n_trials, dtype=bool)
mean_stack = mean_im[:, :, :, np.newaxis]  # (H, W, C, 1)
act_stack = act_img[:, :, np.newaxis]       # (H, W, 1)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    cross_result = align_trials_cross(mean_stack, act_stack, ext_config, keep_trials)
logger.info("Cross-trial alignment: %d valid trials", len(cross_result.valid_trials))

# --- 2d. Source selection ---
valid_pix = np.mean(
    ~np.isnan(cross_result.aligned_mean[:, :, 0, cross_result.valid_trials]),
    axis=2,
) > (1 - ext_config.nan_thresh)

sources = select_sources(
    cross_result.aligned_activity,
    cross_result.valid_trials,
    ext_config,
    valid_pixel_mask=valid_pix,
)
logger.info("Selected %d sources", sources.n_sources)

if sources.n_sources == 0:
    raise RuntimeError("No sources found — check activity_channel and data quality")

# --- 2e. Motion frame detection ---
discard_ds = _detect_motion_frames(adata, ext_config)
logger.info("Discarded frames (ds): %d / %d (%.1f%%)",
            np.sum(discard_ds), len(discard_ds),
            100 * np.sum(discard_ds) / max(1, len(discard_ds)))

# --- 2f. Load full-res activity channel and extract ---
sel_pix_union = np.any(sources.sel_pix, axis=2)

reg_raw = store.load_registered_raw(1)
n_frames = reg_raw.shape[2] // n_ch
movie_raw_4d = reg_raw.reshape(
    reg_raw.shape[0], reg_raw.shape[1], n_frames, n_ch
).transpose(0, 1, 3, 2)
Y_sel = movie_raw_4d[:, :, act_ch, :][sel_pix_union, :].copy()
del reg_raw, movie_raw_4d, reg_ds, movie_4d

Finv = np.ones_like(Y_sel)  # uniform for Bergamo

# Upsample discard mask to full-res
if len(discard_ds) != n_frames:
    discard_full = np.zeros(n_frames, dtype=bool)
    ds_to_full = np.round(np.linspace(0, n_frames - 1, len(discard_ds))).astype(int)
    for d_ix, full_ix in enumerate(ds_to_full):
        if discard_ds[d_ix]:
            discard_full[full_ix] = True
else:
    discard_full = discard_ds[:n_frames]

Y_sel[:, discard_full] = np.nan
logger.info("Extracting: %d pixels, %d sources, %d frames (%.1f Hz)",
            Y_sel.shape[0], sources.n_sources, n_frames, full_hz)

t0 = time.perf_counter()
result = extract_trial(
    Y_sel, Finv,
    sources.rows, sources.cols,
    sel_pix_union,
    ext_config, full_hz,
)
logger.info("NMF extraction done in %.1fs", time.perf_counter() - t0)

# Post-process
denoised = compute_denoised(result.S, ext_config.tau_s, full_hz)
events = censor_frames(result.S, discard_full)
denoised = censor_frames(denoised, discard_full)

valid_snr = result.SNR[~np.isnan(result.SNR)]
logger.info("SNR: median=%.2f, range=[%.2f, %.2f] (%d sources)",
            np.median(valid_snr), np.min(valid_snr), np.max(valid_snr), len(valid_snr))

# ============================================================
# Stage 3: Save everything to Zarr
# ============================================================
logger.info("=== Stage 3: Saving results ===")

store.save_extraction_results(
    trial_idx=1,
    footprints=result.H,
    events=events,
    denoised=denoised,
    ls=censor_frames(result.LS, discard_full),
    f0=result.F0,
    snr=result.SNR,
    discard_frames=discard_full,
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    avg_mean = np.nanmean(
        cross_result.aligned_mean[:, :, :, cross_result.valid_trials], axis=3
    )
    avg_act = np.nanmean(
        cross_result.aligned_activity[:, :, cross_result.valid_trials], axis=2
    )

store.save_summary(
    mean_image=avg_mean,
    activity_image=avg_act,
    source_locations_r=sources.rows,
    source_locations_c=sources.cols,
    sel_pix=sel_pix_union,
    valid_trials=cross_result.valid_trials,
    trial_offsets=cross_result.motion,
    trial_corr_coeffs=cross_result.corr_coeffs,
    params=ext_config.model_dump(),
)

elapsed = time.perf_counter() - t_total
logger.info("=== Pipeline complete in %.1fs ===", elapsed)
logger.info("  Output: %s", zarr_path)
logger.info("  Sources: %d, SNR median: %.2f", sources.n_sources, np.median(valid_snr))
logger.info("  Frames: %d at %.1f Hz (%.1fs recording)", n_frames, full_hz, n_frames / full_hz)

2026-03-12 12:45:34 spine_extraction.io.zarr_store INFO: Opened Zarr store at C:\Users\user\Desktop\iGluSnFR test data\750098\2024-09-24\scan_00003_20240924_112852\registered.zarr
2026-03-12 12:45:34 pipeline INFO: === Stage 1: Motion correction ===
2026-03-12 12:45:34 pipeline INFO: TIFF: scan_00003_20240924_112852.tif
2026-03-12 12:45:34 pipeline INFO: Output: C:\Users\user\Desktop\iGluSnFR test data\750098\2024-09-24\scan_00003_20240924_112852\registered.zarr
2026-03-12 12:45:34 spine_extraction._utils.parallel INFO: Using 1 workers (requested=1, memory-limited=0, trials=1)
2026-03-12 12:45:34 spine_extraction.registration.bergamo INFO: Registering 1 trials with 1 workers
2026-03-12 12:45:34 spine_extraction.registration.bergamo INFO: Aligning trial 1: scan_00003_20240924_112852.tif
2026-03-12 12:45:34 spine_extraction.io.tiff_reader INFO: Opening mmap TIFF: scan_00003_20240924_112852.tif
2026-03-12 12:45:34 spine_extraction.io.tiff_reader INFO: Mmap opened: 46 x 128, 2 ch, 168536 f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-03-12 12:45:35 spine_extraction.registration.template INFO: Creating initial template from 1000 frames
2026-03-12 12:45:35 spine_extraction.registration.template INFO: Selected cluster with 114 frames (mean correlation 0.7737)
2026-03-12 12:45:36 spine_extraction.registration.template INFO: Initial template created: shape (146, 228)
2026-03-12 12:45:36 spine_extraction.registration.bergamo INFO: Will compute 21067 downsampled frames on-the-fly
2026-03-12 12:45:36 spine_extraction.registration.bergamo INFO: Registering 21067 downsampled frames (CPU DFT, CPU interpolation)...
2026-03-12 12:45:38 spine_extraction.registration.bergamo INFO:   Frame 1000 / 21067
2026-03-12 12:45:40 spine_extraction.registration.bergamo INFO:   Frame 2000 / 21067
2026-03-12 12:45:43 spine_extraction.registration.bergamo INFO:   Frame 3000 / 21067
2026-03-12 12:45:45 spine_extraction.registration.bergamo INFO:   Frame 4000 / 21067
2026-03-12 12:45:48 spine_extraction.registration.bergamo INFO:   Frame 50